In [ ]:
# RODANDO SOMENTE COM FUNDEB POR ALUNO

In [15]:
# correlação IDEB e FUNDEB por aluno
df["ideb"].corr(df["ln_fundeb"])

-0.04070051526238921

In [1]:
import pandas as pd
import numpy as np

# Carregar base
df = pd.read_csv("base_final.csv")

# =========================
# LIMPEZA
# =========================

# Converter IDEB para float (muito importante)
df["ideb"] = pd.to_numeric(df["ideb"], errors="coerce")

# Remover valores nulos
df = df.dropna(subset=["ideb", "fundeb", "total"])

# =========================
# FEATURE ENGINEERING
# =========================

# FUNDEB por aluno
df["fundeb_por_aluno"] = df["fundeb"] / df["total"]

# Remover valores inválidos (log não aceita <= 0)
df = df[df["fundeb_por_aluno"] > 0]

# Log do FUNDEB por aluno
df["ln_fundeb"] = np.log(df["fundeb_por_aluno"])

In [ ]:
# Regresão OLS simples

import statsmodels.api as sm


y = df["ideb"]
X = df[["ln_fundeb"]]
X = sm.add_constant(X)


modelo_ols = sm.OLS(y, X).fit()


print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:                   ideb   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     33.51
Date:                Sat, 28 Mar 2026   Prob (F-statistic):           7.20e-09
Time:                        15:54:16   Log-Likelihood:                -27928.
No. Observations:               20197   AIC:                         5.586e+04
Df Residuals:                   20195   BIC:                         5.588e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          6.5307      0.155     42.231      0.0

In [4]:
# modelo pooled OLS

from linearmodels.panel import PooledOLS

# Criar índice de painel
df = df.set_index(["codigo_municipio", "ano"])

y = df["ideb"]
X = df[["ln_fundeb"]]

# Adicionar constante
X = sm.add_constant(X)

modelo_pooled = PooledOLS(y, X)
resultado_pooled = modelo_pooled.fit()

print(resultado_pooled.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                   ideb   R-squared:                        0.0017
Estimator:                  PooledOLS   R-squared (Between):              0.0082
No. Observations:               20197   R-squared (Within):              -0.0432
Date:                Sat, Mar 28 2026   R-squared (Overall):              0.0017
Time:                        15:55:10   Log-likelihood                -2.793e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      33.509
Entities:                        5333   P-value                           0.0000
Avg Obs:                       3.7872   Distribution:                 F(1,20195)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             33.509
                            

In [ ]:
# Modelo com efeitos fixos de município e tempo

from linearmodels.panel import PanelOLS


modelo_fe = PanelOLS(
    df["ideb"],
    df[["ln_fundeb"]],
    entity_effects=True,   # município
    time_effects=True      # ano
)

resultado_fe = modelo_fe.fit(cov_type="clustered", cluster_entity=True)

print(resultado_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                   ideb   R-squared:                        0.0175
Estimator:                   PanelOLS   R-squared (Between):              0.8191
No. Observations:               20197   R-squared (Within):               0.0325
Date:                Sat, Mar 28 2026   R-squared (Overall):              0.8149
Time:                        15:55:51   Log-likelihood                   -6371.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      264.07
Entities:                        5333   P-value                           0.0000
Avg Obs:                       3.7872   Distribution:                 F(1,14860)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             119.96
                            

In [7]:
# Test F (FE VS Pooled)

from linearmodels.panel import compare

comparacao = compare({
    "Pooled": resultado_pooled,
    "FE": resultado_fe
})

print(comparacao)

                  Model Comparison                  
                                Pooled            FE
----------------------------------------------------
Dep. Variable                     ideb          ideb
Estimator                    PooledOLS      PanelOLS
No. Observations                 20197         20197
Cov. Est.                   Unadjusted     Clustered
R-squared                       0.0017        0.0175
R-Squared (Within)             -0.0432        0.0325
R-Squared (Between)             0.0082        0.8191
R-Squared (Overall)             0.0017        0.8149
F-statistic                     33.509        264.07
P-value (F-stat)                0.0000        0.0000
=====================     ============   ===========
const                           6.5307              
                              (42.231)              
ln_fundeb                      -0.0960        0.3630
                             (-5.7887)      (10.953)
======================= ============== =======

In [9]:
# modelo between (entre municípios)

from linearmodels.panel import BetweenOLS

modelo_between = BetweenOLS(
    df["ideb"],
    df[["ln_fundeb"]]
)

resultado_between = modelo_between.fit()

print(resultado_between.summary)

                         BetweenOLS Estimation Summary                          
Dep. Variable:                   ideb   R-squared:                        0.9727
Estimator:                 BetweenOLS   R-squared (Between):              0.9727
No. Observations:                5333   R-squared (Within):              -0.0594
Date:                Sat, Mar 28 2026   R-squared (Overall):              0.9690
Time:                        15:57:57   Log-likelihood                   -7240.2
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                     1.9e+05
Entities:                        5333   P-value                           0.0000
Avg Obs:                       3.7872   Distribution:                  F(1,5332)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):            1.9e+05
                            

In [1]:
# teste de hausman

from linearmodels.panel import PanelOLS, RandomEffects
import statsmodels.api as sm
import numpy as np
from scipy import stats
import pandas as pd

# =========================
# Preparar dados (já com índice)
# =========================
df = pd.read_csv("base_final.csv")

df["ideb"] = pd.to_numeric(df["ideb"], errors="coerce")
df = df.dropna(subset=["ideb", "fundeb", "total"])

df["fundeb_por_aluno"] = df["fundeb"] / df["total"]
df = df[df["fundeb_por_aluno"] > 0]
df["ln_fundeb"] = np.log(df["fundeb_por_aluno"])

df = df.set_index(["codigo_municipio", "ano"])

# =========================
# Variáveis
# =========================
y = df["ideb"]
X = df[["ln_fundeb"]]

# =========================
# MODELO FE
# =========================
modelo_fe = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_fe = modelo_fe.fit()

# =========================
# MODELO RE
# =========================
modelo_re = RandomEffects(y, X)
res_re = modelo_re.fit()

# =========================
# TESTE DE HAUSMAN
# =========================

# Coeficientes
b_fe = res_fe.params
b_re = res_re.params

# Diferença
diff = b_fe - b_re

# Covariâncias
cov_fe = res_fe.cov
cov_re = res_re.cov

# Estatística de Hausman
stat = np.dot(np.dot(diff.T, np.linalg.inv(cov_fe - cov_re)), diff)

# Graus de liberdade
df_h = diff.shape[0]

# p-valor
p_value = 1 - stats.chi2.cdf(stat, df_h)

print("Hausman Test")
print(f"Chi2: {stat}")

Hausman Test
Chi2: 107.58132750641873


In [2]:
# teste de heteroscedasticidade

from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm

# =========================
# OLS já rodado antes
# =========================
y = df.reset_index()["ideb"]
X = df.reset_index()[["ln_fundeb"]]
X = sm.add_constant(X)

modelo_ols = sm.OLS(y, X).fit()

# =========================
# TESTE BREUSCH-PAGAN
# =========================
bp_test = het_breuschpagan(modelo_ols.resid, modelo_ols.model.exog)

labels = ["LM Statistic", "LM p-value", "F Statistic", "F p-value"]

for i in range(len(labels)):
    print(f"{labels[i]}: {bp_test[i]}")

LM Statistic: 4.274148470943048
LM p-value: 0.03869634821935388
F Statistic: 4.274629834791046
F p-value: 0.03869809683441177


In [ ]:
# corrigindo erro padrão

from linearmodels.panel import PanelOLS

modelo_fe = PanelOLS(
    df["ideb"],
    df[["ln_fundeb"]],
    entity_effects=True,
    time_effects=True
)

resultado_fe = modelo_fe.fit(cov_type="clustered", cluster_entity=True)
print(resultado_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                   ideb   R-squared:                        0.0175
Estimator:                   PanelOLS   R-squared (Between):              0.8191
No. Observations:               20197   R-squared (Within):               0.0325
Date:                Sat, Mar 28 2026   R-squared (Overall):              0.8149
Time:                        16:08:55   Log-likelihood                   -6371.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      264.07
Entities:                        5333   P-value                           0.0000
Avg Obs:                       3.7872   Distribution:                 F(1,14860)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             119.96
                            

In [11]:
# OLS robusto

import statsmodels.api as sm

# Resetar índice (IMPORTANTE)
df_reset = df.reset_index()

# Variáveis
y = df_reset["ideb"]
X = df_reset[["ln_fundeb"]]

# Constante
X = sm.add_constant(X)

# OLS normal
modelo_ols_robusto = sm.OLS(y, X).fit(cov_type='HC1')




In [13]:
# resultado do modelo between (entre municípios)

from linearmodels.panel import BetweenOLS

# Modelo Between
modelo_between = BetweenOLS(
    df["ideb"],
    df[["ln_fundeb"]]
)

# Salvar resultado
resultado_between = modelo_between.fit()

# Ver resumo
print(resultado_between.summary)

                         BetweenOLS Estimation Summary                          
Dep. Variable:                   ideb   R-squared:                        0.9727
Estimator:                 BetweenOLS   R-squared (Between):              0.9727
No. Observations:                5333   R-squared (Within):              -0.0594
Date:                Sat, Mar 28 2026   R-squared (Overall):              0.9690
Time:                        16:17:17   Log-likelihood                   -7240.2
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                     1.9e+05
Entities:                        5333   P-value                           0.0000
Avg Obs:                       3.7872   Distribution:                  F(1,5332)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):            1.9e+05
                            

In [14]:
# tabela de comparação entre modelos

import pandas as pd

# Coeficientes
coef_ols = modelo_ols_robusto.params["ln_fundeb"]
se_ols = modelo_ols_robusto.bse["ln_fundeb"]

coef_fe = resultado_fe.params["ln_fundeb"]
se_fe = resultado_fe.std_errors["ln_fundeb"]

coef_between = resultado_between.params["ln_fundeb"]
se_between = resultado_between.std_errors["ln_fundeb"]

# R²
r2_ols = modelo_ols_robusto.rsquared
r2_fe = resultado_fe.rsquared_within
r2_between = resultado_between.rsquared

# Criar tabela
tabela = pd.DataFrame({
    "Variável": ["ln(FUNDEB por aluno)"],
    "OLS": [f"{coef_ols:.3f} ({se_ols:.3f})"],
    "FE": [f"{coef_fe:.3f} ({se_fe:.3f})"],
    "Between": [f"{coef_between:.3f} ({se_between:.3f})"]
})

# Adicionar estatísticas
stats = pd.DataFrame({
    "Variável": ["R²", "Observações"],
    "OLS": [f"{r2_ols:.3f}", len(df)],
    "FE": [f"{r2_fe:.3f}", len(df)],
    "Between": [f"{r2_between:.3f}", resultado_between.nobs]
})

tabela_final = pd.concat([tabela, stats])

print(tabela_final)

               Variável             OLS             FE        Between
0  ln(FUNDEB por aluno)  -0.096 (0.017)  0.363 (0.033)  0.602 (0.001)
0                    R²           0.002          0.033          0.973
1           Observações           20197          20197           5333
